# Gravitino filesets

Filesets let Gravitino manage collections of files on storage (here, HDFS) as first-class metadata, the same way it manages tables. This notebook walks the full lifecycle with the Gravitino Python client: create a metalake and a fileset catalog, add a schema, create both a **managed** and an **external** fileset, list and load them, then tear everything down, checking the underlying HDFS paths at each step.

The key distinction it demonstrates: dropping a *managed* fileset deletes its storage location, while dropping an *external* fileset leaves the storage in place.

## Set up the HDFS client

Install the `hdfs` package first (note `%pip`, which installs into this notebook's kernel), then connect to the playground's HDFS NameNode. We list and clear `/user/gravitino` so the example starts from a clean state.

In [ ]:
# Ensure build tools are present first: the hdfs package builds from source and
# needs setuptools/wheel, which are not always available in the image's build env.
%pip install -q --upgrade pip setuptools wheel
%pip install -q hdfs

In [ ]:
from hdfs import InsecureClient
import os

# Connect to the playground's HDFS NameNode.
hdfs_client = InsecureClient("http://hive:50070", user='root')

# Start from a clean working directory. On a fresh HDFS it may not exist yet,
# which is fine; list/delete are guarded so this is safe on first or repeat runs.
try:
    print("existing contents:", hdfs_client.list('/user/gravitino'))
    hdfs_client.delete("/user/gravitino", recursive=True)
    print("cleared /user/gravitino")
except Exception:
    print("/user/gravitino does not exist yet (clean start)")

# Ensure the working directory exists for the steps below.
hdfs_client.makedirs('/user/gravitino')
print("ready: /user/gravitino")

## Connect to Gravitino and create a metalake

Install the Gravitino Python client (matching the 1.3.0 server), then use the admin client to create a metalake. A metalake is the top-level container that holds catalogs.

In [ ]:
%pip install -q apache-gravitino==1.3.0

In [ ]:
from typing import Dict, List
from gravitino import NameIdentifier, GravitinoAdminClient, GravitinoClient, Catalog, Fileset, FilesetChange
import os

# The admin client manages metalakes.
gravitino_admin_client = GravitinoAdminClient(uri="http://gravitino:8090")

# Create (or load, if a previous run left it) a metalake to hold our catalog.
metalake_name = "default"
try:
    metalake = gravitino_admin_client.load_metalake(metalake_name)
    print(f"metalake {metalake_name}: already exists (loaded)")
except Exception:
    metalake = gravitino_admin_client.create_metalake(
        name=metalake_name, comment="metalake comment", properties={})
    print(f"metalake {metalake_name}: created")
print(metalake)

Open a metalake-scoped client. All catalog, schema, and fileset operations below go through this client.

In [ ]:
gravitino_client = GravitinoClient(uri="http://gravitino:8090", metalake_name=metalake_name)

Confirm the metalake exists by listing all metalakes on the server.

In [ ]:
from gravitino import GravitinoMetalake

metalake_list: List[GravitinoMetalake] = gravitino_admin_client.list_metalakes()
print(metalake_list)

## Create a fileset catalog

Create a catalog of type `FILESET` backed by the `hadoop` provider. This is the catalog that will hold our filesets.

In [ ]:
catalog_name = "catalog"
try:
    catalog = gravitino_client.load_catalog(name=catalog_name)
    print(f"catalog {catalog_name}: already exists (loaded)")
except Exception:
    catalog = gravitino_client.create_catalog(
        name=catalog_name,
        catalog_type=Catalog.Type.FILESET,
        provider="hadoop",
        comment="",
        properties={})
    print(f"catalog {catalog_name}: created")
print(catalog)

Load the catalog back to confirm it was created.

In [ ]:
catalog = gravitino_client.load_catalog(name=catalog_name)
print(catalog)

## Create a schema

A schema groups filesets and maps to a storage location. We create the schema, then verify its directory was created in HDFS.

In [ ]:
schema_name = "schema"
schema_path = "/user/gravitino/" + schema_name
schema_hdfs_path = f"hdfs://hive:9000{schema_path}"

try:
    catalog.as_schemas().load_schema(schema_name=schema_name)
    print(f"schema {schema_name}: already exists (loaded)")
except Exception:
    catalog.as_schemas().create_schema(
        schema_name=schema_name, comment="", properties={"location": schema_hdfs_path})
    print(f"schema {schema_name}: created")

# Verify the schema's storage location exists in HDFS.
try:
    info = hdfs_client.status(schema_path)
    print(f"Success: storage location {schema_path} exists.")
    print("Details:", info)
except Exception:
    print(f"Failed: storage location {schema_path} was not created.")

## Create a managed fileset

A **managed** fileset: Gravitino owns its storage lifecycle. Creating it provisions the HDFS location; dropping it later will delete that location.

In [ ]:
managed_fileset_name = "managed_fileset"
managed_fileset_path = "/user/gravitino/" + schema_name + "/" + managed_fileset_name
managed_fileset_hdfs_path = f"hdfs://hive:9000{managed_fileset_path}"

managed_fileset_ident: NameIdentifier = NameIdentifier.of(schema_name, managed_fileset_name)
catalog.as_fileset_catalog().create_fileset(
    ident=managed_fileset_ident,
    fileset_type=Fileset.Type.MANAGED,
    comment="",
    storage_location=managed_fileset_hdfs_path,
    properties={})

# Verify the managed fileset's storage location exists in HDFS.
try:
    info = hdfs_client.status(managed_fileset_path)
    print(f"Success: storage location {managed_fileset_path} was created.")
    print("Details:", info)
except Exception:
    print(f"Failed: storage location {managed_fileset_path} was not created.")

## Create an external fileset

An **external** fileset points at a storage location you provision yourself. Gravitino tracks it but does not own its lifecycle; dropping it later will leave the storage in place. Here we create the HDFS directory first, then register the external fileset over it.

In [ ]:
external_fileset_name = "external_fileset"
external_fileset_path = "/user/gravitino/" + schema_name + "/" + external_fileset_name
external_fileset_hdfs_path = f"hdfs://hive:9000{external_fileset_path}"

# Provision the storage location ourselves, before registering the fileset.
hdfs_client.makedirs(external_fileset_path)
try:
    info = hdfs_client.status(external_fileset_path)
    print(f"Success: storage location {external_fileset_path} was created.")
    print("Details:", info)
except Exception:
    print(f"Failed: storage location {external_fileset_path} was not created.")

# Register an external fileset over the existing location.
external_fileset_ident: NameIdentifier = NameIdentifier.of(schema_name, external_fileset_name)
catalog.as_fileset_catalog().create_fileset(
    ident=external_fileset_ident,
    fileset_type=Fileset.Type.EXTERNAL,
    comment="",
    storage_location=external_fileset_hdfs_path,
    properties={})

## List and load filesets

List all filesets in the schema, then load each one back individually.

In [ ]:
catalog = gravitino_client.load_catalog(name=catalog_name)
fileset_list: List[NameIdentifier] = catalog.as_fileset_catalog().list_filesets(
    namespace=managed_fileset_ident.namespace())
print(fileset_list)

In [ ]:
# Load the managed fileset.
managed_fileset = gravitino_client.load_catalog(name=catalog_name).as_fileset_catalog().load_fileset(ident=managed_fileset_ident)
print(managed_fileset)

In [ ]:
# Load the external fileset.
external_fileset = gravitino_client.load_catalog(name=catalog_name).as_fileset_catalog().load_fileset(ident=external_fileset_ident)
print(external_fileset)

## Drop filesets and observe the storage difference

This is the payoff. Dropping the **managed** fileset deletes its HDFS location; dropping the **external** fileset leaves its location intact.

In [ ]:
# Drop the managed fileset. Its HDFS location should be deleted.
catalog.as_fileset_catalog().drop_fileset(ident=managed_fileset_ident)

try:
    info = hdfs_client.status(managed_fileset_path)
    print(f"Unexpected: storage location {managed_fileset_path} was not deleted.")
except Exception:
    print(f"Success: storage location {managed_fileset_path} was deleted (managed).")

In [ ]:
# Drop the external fileset. Its HDFS location should be preserved.
catalog.as_fileset_catalog().drop_fileset(ident=external_fileset_ident)

try:
    info = hdfs_client.status(external_fileset_path)
    print(f"Success: storage location {external_fileset_path} preserved (external).")
except Exception:
    print(f"Unexpected: storage location {external_fileset_path} was deleted.")

## Clean up

Drop the schema, catalog, and metalake to return the server to its starting state.

In [ ]:
# Drop the schema (cascade removes remaining contents).
catalog.as_schemas().drop_schema(schema_name=schema_name, cascade=True)

try:
    info = hdfs_client.status(schema_path)
    print(f"Unexpected: storage location {schema_path} was not deleted.")
except Exception:
    print(f"Success: storage location {schema_path} was deleted.")

In [ ]:
# Drop the catalog.
result = gravitino_client.drop_catalog(name=catalog_name, force=True)
print(result)

In [ ]:
# Drop the metalake.
result = gravitino_admin_client.drop_metalake(metalake_name, force=True)
print(result)